In [74]:
import pandas as pd
import numpy as np

movies_df = pd.read_csv('data/movies.csv')
movies = movies_df.copy()

ratings = pd.read_csv('data/ratings.csv')
ratings = ratings.copy()

#tags_df = pd.read_csv('data/tags.csv')
#tags = tags_df.copy()

Filtering ratings

In [75]:
lowest_acceptable_mean_rating_score = 2.0 # Set to zero to disable filtering
lowest_acceptable_rating_count = 100

ratings_grouped = ratings.groupby('movieId')['rating'].mean().sort_values()
good_movie_scores = ratings_grouped[ratings_grouped >= lowest_acceptable_mean_rating_score].index
ratings_filtered = ratings[ratings["movieId"].isin(good_movie_scores)]

ratings_per_movie = ratings.groupby('movieId')['rating'].count()
popular_movie = ratings_per_movie[ratings_per_movie >= lowest_acceptable_rating_count].index
ratings_filtered = ratings_filtered[ratings_filtered["movieId"].isin(popular_movie)]

Splitting movie genres string with on-hot encoder 

In [76]:
genre_dummies = movies["genres"].str.get_dummies("|")
movies_genre_one_hot = pd.concat([movies, genre_dummies], axis=1)
movies_genre_one_hot = movies_genre_one_hot.drop(columns="genres")


Filtering movies without genres

In [77]:
movies_filtered = movies_genre_one_hot[movies_genre_one_hot["(no genres listed)"] == 0]
movies_filtered = movies_filtered.drop(columns="(no genres listed)")
all_movies_genre_matrix = movies_filtered.drop(columns=["movieId", "title"])
movies_filtered.info()

<class 'pandas.DataFrame'>
Index: 79477 entries, 0 to 86536
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   movieId      79477 non-null  int64
 1   title        79477 non-null  str  
 2   Action       79477 non-null  int64
 3   Adventure    79477 non-null  int64
 4   Animation    79477 non-null  int64
 5   Children     79477 non-null  int64
 6   Comedy       79477 non-null  int64
 7   Crime        79477 non-null  int64
 8   Documentary  79477 non-null  int64
 9   Drama        79477 non-null  int64
 10  Fantasy      79477 non-null  int64
 11  Film-Noir    79477 non-null  int64
 12  Horror       79477 non-null  int64
 13  IMAX         79477 non-null  int64
 14  Musical      79477 non-null  int64
 15  Mystery      79477 non-null  int64
 16  Romance      79477 non-null  int64
 17  Sci-Fi       79477 non-null  int64
 18  Thriller     79477 non-null  int64
 19  War          79477 non-null  int64
 20  Western      79477 non

Merging filtered movies and ratings dataframe

In [78]:
movies_ratings_merge = ratings_filtered.merge(movies_filtered, on="movieId", how="inner")
movies_ratings_merge.shape

(32911847, 24)

Rekommender Contet
"identify similarities between movies based on genres based filtering KNN och cosine similarity"
"one-hot encoding on genres and a KNN-Transform with cosine similarity"

In [79]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_Cosine(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    if matches.empty:
       return "Movie not found"
    
    idx = matches.index[0]
    all_movies_genre = movies_filtered.drop(columns=["movieId", "title"])
    input_movie_genre = all_movies_genre.iloc[idx].values.reshape(1, -1)

    # Inputfilm vs all movies -> Similarity
    similarity_scores = cosine_similarity(input_movie_genre, all_movies_genre).flatten()

    similarity_scores_df = pd.DataFrame({"similarity_score": similarity_scores}) # Array from cosine_similarity to pandas dataframe
    similarity_scores_sorted_df = similarity_scores_df.sort_values(by="similarity_score", ascending=False)
    similarity_scores_sorted_top5_df = similarity_scores_sorted_df.iloc[1:n+1]   # Removing self
    similarity_df = similarity_scores_sorted_top5_df                             # shortening name

    similarity_df["title"] = movies_filtered.loc[similarity_df.index, "title"]  # Replaces movieId with title 
    similarity_df = similarity_df.loc[:, ['title', 'similarity_score']]         # Changing order of columns
    return similarity_df
recommend_Cosine("Toy Story (1995)")

,title,similarity_score
78338,Emergency (2017),1.0
3654,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
3913,"Emperor's New Groove, The (2000)",1.0
4781,"Monsters, Inc. (2001)",1.0
9951,DuckTales: The Movie - Treasure of the Lost La...,1.0


In [96]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_movies_genre_matrix)

#def recommend_KNN(movie_title, n=5):
    #return result

#recommend_KNN("Toy Story (1995)")

movie_title = "Toy Story (1995)"
n=5

matches = movies_filtered[movies_filtered["title"] == movie_title]
#if matches.empty:
#   return "Movie not found"

idx = matches.index[0]
input_movie_genre = all_movies_genre_matrix.iloc[[idx]]


knn_distances, knn_indices = knn.kneighbors(input_movie_genre, n_neighbors=n+1)
# platta arrays
knn_indices = knn_indices.flatten()[1:]      # hoppa över sig själv
knn_distances = knn_distances.flatten()[1:]

knn_similarity_scores = 1 - knn_distances    # cosine distance → similarity

result = movies_filtered.iloc[knn_indices][["title"]].copy()
result["similarity_score"] = knn_similarity_scores

knn_distances


array([0., 0., 0., 0., 0.])